# exp072 (M3'): eca_nfnet_l0 + 20s + Babych init + Hybrid V2 + R3 3-fold

**M-R3 mel軸 core (paradigm fix v2)**

**Spec (V3 = paradigm fix)**:
  - Backbone: `eca_nfnet_l0.ra2_in1k`
  - Init: ★ Babych BC25 iter3 eca_nfnet_l0 ckpt (★ key overlap 192/192 確認済)
  - Mel: 20s × 224 × 4096 × 1252 (Babych spec)
  - Output: 234 classes
  - Loss: BCE clip + framewise max + Hybrid V2 pseudo distill
  - **Aug**: ★ MixUp Beta(0.4) × P=0.5 ★ (M7 流、V1 の 100% fixed-blend MixUp を改善)
  - **drop_path**: ★ 0.10 ★ (V1 の 0.15 から下げ、random head に優しく)
  - **Pseudo**: ★ exp069c V3 `pseudo_hybrid_234.npz` ★ (Babych rank-pct と exp048 rank-pct で scale 統一済)
  - Power transform k = 1.54
  - **Val protocol**: ★ labeled SC val (M1/M2 同 protocol、LB 相関 proven) ★ (V1 の focal val から変更)
  - 3-fold StratifiedKFold + 20 epoch

**V1 (失敗) からの修正**:
| 項目 | V1 (val 0.53 stuck) | V3 (M3') |
|---|---|---|
| Pseudo | Hybrid V1 (scale mismatch 10x) | Hybrid V2 (rank-pct unified) |
| MixUp | fixed 0.5 × P=1.0 | Beta(0.4) × P=0.5 |
| drop_path | 0.15 | 0.10 |
| Val | focal val (234 class single-label) | labeled SC val (multi-label) |

**Output**: m3_fold{0,1,2}_ckpt_best.pth + ONNX export per fold + history JSON


In [ ]:
!pip install onnxruntime --quiet
import sys
print(f"Python: {sys.version[:50]}")


In [ ]:
import os, time, json, gc, math, random, warnings
warnings.filterwarnings("ignore")  # ★ V3: suppress FutureWarning / UserWarning noise
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import librosa
import timm
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score
import tqdm.auto as tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, torch {torch.__version__}, timm {timm.__version__}")
START = time.time()


In [ ]:
# M3' CFG (V4: V3 paradigm fix + grad_clip + LR 安全化)
SR = 32_000
WINDOW_SEC = 20                     # ★ Babych 20s
WINDOW_SAMPLES = SR * WINDOW_SEC
N_WINDOWS_PSEUDO = 12               # pseudo file 12 chunks (test_soundscape format)
WINDOW_SEC_PSEUDO = 5

# Mel (Babych spec)
N_MELS = 224
N_FFT = 4096
HOP_LENGTH = 1252
F_MIN = 0
F_MAX = 16000
TOP_DB = 80

# Training
N_FOLDS = 3
N_EPOCHS = 20
BATCH_SIZE = 24
LR = 3e-4                           # ★ V4: 5e-4 → 3e-4 (V3 で ep3 NaN 起きたので保守化)
LR_MIN = 1e-6
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
DROP_PATH = 0.10
GRAD_CLIP = 1.0                     # ★ V4: gradient explosion 防止 (V3 NaN 主因)

# Pseudo (Hybrid V2)
PSEUDO_WEIGHT = 0.20
PSEUDO_POWER_K = 1.54
PSEUDO_LOSS_WEIGHT = 0.5

# Aug V3: M7 流 Beta MixUp
MIXUP_MODE = "beta"
MIXUP_BETA_ALPHA = 0.4
MIXUP_P = 0.5
SPECAUG_FREQ = 10
SPECAUG_TIME = 10

# Validation
STEP_LOG_INTERVAL = 100
NS22_K = 22

SEED = 42

# Paths
_data_path_candidates = [
    "/kaggle/input/competitions/birdclef-2026",
    "/kaggle/input/birdclef-2026",
]
DATA_PATH = None
for _p in _data_path_candidates:
    if Path(_p).exists():
        DATA_PATH = _p; break
assert DATA_PATH is not None

TRAIN_CSV = Path(DATA_PATH) / "train.csv"
TRAIN_AUDIO_DIR = Path(DATA_PATH) / "train_audio"
TRAIN_SC_DIR = Path(DATA_PATH) / "train_soundscapes"
TRAIN_SC_LABELS_CSV = Path(DATA_PATH) / "train_soundscapes_labels.csv"
TAXONOMY_CSV = Path(DATA_PATH) / "taxonomy.csv"
SAMPLE_SUB_PATH = Path(DATA_PATH) / "sample_submission.csv"

# Babych nfnet_l0 weight
BABYCH_DIR = None
for _p in ["/kaggle/input/birdclef2025-1st-place-ensemble", "/kaggle/input/datasets/nikitababich/birdclef2025-1st-place-ensemble"]:
    if Path(_p).exists():
        BABYCH_DIR = Path(_p); break
assert BABYCH_DIR is not None
BABYCH_NFNET_CKPT = None
for f in BABYCH_DIR.glob("eca_nfnet_l0*.pt"):
    BABYCH_NFNET_CKPT = f; break
assert BABYCH_NFNET_CKPT is not None
print(f"Babych ckpt: {BABYCH_NFNET_CKPT.name}")

# exp069c V3 pseudo (Hybrid V2)
EXP069C_DIR = None
for _p in ["/kaggle/input/birdclef2026-exp069c-hybrid-merge", "/kaggle/input/notebooks/maekeso/birdclef2026-exp069c-hybrid-merge"]:
    if Path(_p).exists():
        EXP069C_DIR = Path(_p); break
assert EXP069C_DIR is not None, "exp069c output not attached"
print(f"exp069c dir: {EXP069C_DIR}")

OUT_DIR = Path("/kaggle/working")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n[M3' V4 CFG]")
print(f"  Backbone: eca_nfnet_l0 + Babych iter3 init")
print(f"  Mel: {WINDOW_SEC}s × {N_MELS} × {N_FFT} × {HOP_LENGTH}")
print(f"  MixUp: Beta({MIXUP_BETA_ALPHA}) × P={MIXUP_P}")
print(f"  drop_path: {DROP_PATH}")
print(f"  LR peak: {LR}  (V3 5e-4 → V4 3e-4 NaN safety)")
print(f"  ★ GRAD_CLIP: {GRAD_CLIP}  (★ V4: NaN 対策、M1 spec 準拠)")
print(f"  Pseudo: hybrid V2 power k={PSEUDO_POWER_K}")
print(f"  Val: labeled SC (M1/M2 protocol)")


In [ ]:
# Load BC26 taxonomy + train.csv + labeled SC val
taxo = pd.read_csv(TAXONOMY_CSV)
PRIMARY_LABELS = taxo["primary_label"].astype(str).tolist()
N_CLASSES = len(PRIMARY_LABELS)
assert N_CLASSES == 234
LABEL2IDX = {label: i for i, label in enumerate(PRIMARY_LABELS)}

label_to_taxon = dict(zip(taxo["primary_label"].astype(str), taxo["class_name"].astype(str)))
TAXON_MASKS = {
    t: np.array([i for i, lbl in enumerate(PRIMARY_LABELS) if label_to_taxon.get(lbl, "") == t])
    for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]
}
print(f"Taxon counts: " + " ".join([f"{t}={len(m)}" for t, m in TAXON_MASKS.items()]))

# train.csv
train_df = pd.read_csv(TRAIN_CSV)
train_df["primary_label"] = train_df["primary_label"].astype(str)
train_df = train_df[train_df["primary_label"].isin(LABEL2IDX)].reset_index(drop=True)
train_df["exists"] = train_df["filename"].map(lambda fn: (TRAIN_AUDIO_DIR / fn).exists())
train_df = train_df[train_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
print(f"train.csv: {len(train_df)} focal recordings")

# 3-fold StratifiedKFold by primary_label (focal train split のみ、val は labeled SC を使用)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for fi, (_, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    train_df.loc[val_idx, "fold"] = fi
print(f"Focal fold dist: {train_df['fold'].value_counts().sort_index().to_dict()}")

# ★ V3: Load labeled SC for validation (M1/M2 同 protocol)
if TRAIN_SC_LABELS_CSV.exists():
    sc_labels_raw = pd.read_csv(TRAIN_SC_LABELS_CSV).drop_duplicates()
    if sc_labels_raw["start"].dtype == object:
        sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)
    else:
        sc_labels_raw["start_sec"] = sc_labels_raw["start"].astype(int)
    sc_meta = sc_labels_raw[["filename", "start_sec"]].drop_duplicates().reset_index(drop=True)
    if "site" in sc_labels_raw.columns:
        site_map = sc_labels_raw.groupby("filename")["site"].first().to_dict()
        sc_meta["site"] = sc_meta["filename"].map(site_map).fillna("UNK")
    else:
        sc_meta["site"] = "UNK"
    Y_SC = np.zeros((len(sc_meta), N_CLASSES), dtype=np.float32)
    for i, row in sc_meta.iterrows():
        matches = sc_labels_raw[(sc_labels_raw["filename"] == row["filename"]) & (sc_labels_raw["start_sec"] == row["start_sec"])]
        for _, m in matches.iterrows():
            for lbl in str(m["primary_label"]).split(";"):
                lbl = lbl.strip()
                if lbl in LABEL2IDX:
                    Y_SC[i, LABEL2IDX[lbl]] = 1.0
    sc_files = sc_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
    gkf = GroupKFold(n_splits=N_FOLDS)
    sc_files["fold"] = -1
    for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
        sc_files.loc[sc_files.index[val_idx], "fold"] = fold
    file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
    sc_meta["fold"] = sc_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
    non_s22_mask_sc = (sc_meta["site"].values != "S22")
    print(f"sc_meta: {len(sc_meta)} chunks, sc_files: {len(sc_files)} files")
    print(f"  SC fold dist: {sc_meta['fold'].value_counts().sort_index().to_dict()}")
else:
    sc_meta = pd.DataFrame(columns=["filename", "start_sec", "site", "fold"])
    Y_SC = np.zeros((0, N_CLASSES), dtype=np.float32)
    non_s22_mask_sc = np.zeros(0, dtype=bool)
    print("[WARN] labeled SC not found, val will be empty")


In [ ]:
# Load Hybrid pseudo (from exp069c)
pseudo_npz = dict(np.load(EXP069C_DIR / "pseudo_hybrid_234.npz", allow_pickle=True))
pseudo_probs = pseudo_npz["probs"].astype(np.float32)   # (n_files, 12, 234) probabilities
pseudo_file_ids = pseudo_npz["file_ids"]
print(f"Hybrid pseudo: {pseudo_probs.shape}, n_files={len(pseudo_file_ids)}")

# Apply power transform k=1.54 (Babych iter2)
pseudo_probs_k = pseudo_probs ** PSEUDO_POWER_K
print(f"Power transform k={PSEUDO_POWER_K} applied, mean={pseudo_probs_k.mean():.4f}")

# Build pseudo dataset index (each (file_id, chunk_idx) row is a sample)
N_PSEUDO_FILES = len(pseudo_file_ids)
N_PSEUDO_CHUNKS = N_PSEUDO_FILES * N_WINDOWS_PSEUDO   # 12 chunks per file
print(f"Total pseudo chunks: {N_PSEUDO_CHUNKS}")


In [ ]:
# Babych SED architecture (234 class output)
def gem_freq(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), 1)).pow(1.0 / p)


class GeMFreq(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return gem_freq(x, p=self.p, eps=self.eps)


class AttHead(nn.Module):
    def __init__(self, in_chans, p=0.5, num_class=N_CLASSES, hidden_dim=512):
        super().__init__()
        self.pooling = GeMFreq()
        self.dense_layers = nn.Sequential(
            nn.Dropout(p / 2),
            nn.Linear(in_chans, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p),
        )
        self.attention = nn.Conv1d(hidden_dim, num_class, kernel_size=1, bias=True)
        self.fix_scale = nn.Conv1d(hidden_dim, num_class, kernel_size=1, bias=True)

    def forward(self, feat):
        feat = self.pooling(feat).squeeze(-2).permute(0, 2, 1)
        feat = self.dense_layers(feat).permute(0, 2, 1)
        framewise_logit = self.fix_scale(feat)
        return {"framewise_logit": framewise_logit}


class NormalizeMelSpec(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, X):
        mean = X.mean((1, 2), keepdim=True)
        std = X.std((1, 2), keepdim=True)
        Xstd = (X - mean) / (std + self.eps)
        norm_max = torch.amax(Xstd, dim=(1, 2), keepdim=True)
        norm_min = torch.amin(Xstd, dim=(1, 2), keepdim=True)
        return (Xstd - norm_min) / (norm_max - norm_min + self.eps)


class SpecFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            T.MelSpectrogram(sample_rate=SR, normalized=True, n_fft=N_FFT,
                             hop_length=HOP_LENGTH, win_length=N_FFT,
                             f_max=F_MAX, n_mels=N_MELS, f_min=F_MIN),
            T.AmplitudeToDB(top_db=TOP_DB),
        )
        self.norm = NormalizeMelSpec()
    def forward(self, x):
        return self.norm(self.feature_extractor(x))


class CLEFClassifierSED(nn.Module):
    def __init__(self, backbone_name="eca_nfnet_l0.ra2_in1k", num_classes=N_CLASSES, drop_path_rate=DROP_PATH):
        super().__init__()
        self.mel_spectr_generator = SpecFeatureExtractor()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, features_only=True,
            in_chans=3, drop_path_rate=drop_path_rate,
        )
        backbone_dim = self.backbone.feature_info.channels()[-1]
        self.head = AttHead(in_chans=backbone_dim, num_class=num_classes)

    def forward(self, wav, return_framewise=False):
        spec = self.mel_spectr_generator(wav)
        spec3 = torch.stack([spec, spec, spec], 1)
        feat = self.backbone(spec3)[-1]
        head_output = self.head(feat)
        framewise_logit = head_output["framewise_logit"]
        clip_logit = framewise_logit.max(dim=-1).values
        if return_framewise:
            return clip_logit, framewise_logit.permute(0, 2, 1)
        return clip_logit


# ============================================================
# ★ Diagnostic: Babych ckpt key 構造を dump、load_state_dict msg を verbose 化
# ============================================================
print(f"\n=== Babych ckpt diagnostic ===")
_babych_state = torch.load(str(BABYCH_NFNET_CKPT), weights_only=True, map_location="cpu")
print(f"Total keys: {len(_babych_state)}")
_prefix_counter = {}
for k in _babych_state.keys():
    pre = k.split(".", 1)[0]
    _prefix_counter[pre] = _prefix_counter.get(pre, 0) + 1
print(f"Top-level prefixes: {_prefix_counter}")
print(f"Sample keys (first 10):")
for k in list(_babych_state.keys())[:10]:
    v = _babych_state[k]
    sh = tuple(v.shape) if hasattr(v, "shape") else "(scalar)"
    print(f"  {k}: {sh}")
print(f"Sample keys (last 5):")
for k in list(_babych_state.keys())[-5:]:
    v = _babych_state[k]
    sh = tuple(v.shape) if hasattr(v, "shape") else "(scalar)"
    print(f"  {k}: {sh}")

# Build our model and dump its keys
_our_model = CLEFClassifierSED()
_our_state = _our_model.state_dict()
print(f"\nOur model keys: {len(_our_state)}")
_our_prefix_counter = {}
for k in _our_state.keys():
    pre = k.split(".", 1)[0]
    _our_prefix_counter[pre] = _our_prefix_counter.get(pre, 0) + 1
print(f"Our top-level prefixes: {_our_prefix_counter}")

# Find overlapping keys (exact match)
_babych_keys = set(_babych_state.keys())
_our_keys = set(_our_state.keys())
_overlap = _babych_keys & _our_keys
print(f"\nExact key overlap: {len(_overlap)} / {len(_our_keys)} our keys")
if len(_overlap) == 0:
    print("  ★★★ ZERO overlap — Babych key prefix mismatch ★★★")
    # Try common remap patterns
    print("\nAttempting remap:")
    for remap_prefix in ["model.", "module."]:
        _remapped = {k[len(remap_prefix):]: v for k, v in _babych_state.items() if k.startswith(remap_prefix)}
        _ov2 = set(_remapped.keys()) & _our_keys
        print(f"  strip '{remap_prefix}': overlap = {len(_ov2)}")
del _our_model
gc.collect()


def make_model_with_babych_init():
    model = CLEFClassifierSED()
    state = torch.load(str(BABYCH_NFNET_CKPT), weights_only=True, map_location="cpu")

    # Try key prefix variations to maximize transfer
    raw_keys = set(state.keys())
    our_keys = set(model.state_dict().keys())
    direct_overlap = len(raw_keys & our_keys)

    if direct_overlap == 0:
        # Try stripping common wrapper prefixes
        for pre in ["model.", "module.", "net."]:
            stripped = {k[len(pre):]: v for k, v in state.items() if k.startswith(pre)}
            if len(set(stripped.keys()) & our_keys) > 0:
                print(f"  [remap] stripping '{pre}' prefix")
                state = stripped
                break

    # Filter: backbone + mel only (skip head with 206-class mismatch)
    fstate = {k: v for k, v in state.items()
              if k.startswith("backbone.") or k.startswith("mel_spectr_generator.")}
    msg = model.load_state_dict(fstate, strict=False)
    print(f"  [Babych load] applied keys: {len(fstate)}")
    print(f"  [Babych load] missing_keys: {len(msg.missing_keys)} (e.g. {msg.missing_keys[:3]})")
    print(f"  [Babych load] unexpected_keys: {len(msg.unexpected_keys)} (e.g. {msg.unexpected_keys[:3]})")
    # Sanity: backbone first conv weight should differ from imagenet init
    return model


_tmp = make_model_with_babych_init()
print(f"\nM3 model: {sum(p.numel() for p in _tmp.parameters())/1e6:.1f}M params (Babych init)")
del _tmp; gc.collect()


In [ ]:
# Datasets
class FocalDS(Dataset):
    def __init__(self, df, label2idx, train_audio_dir, train_mode=True):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.train_audio_dir = Path(train_audio_dir)
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df)

    def load_audio(self, filename):
        try:
            y, _ = librosa.load(str(self.train_audio_dir / filename), sr=SR, mono=True)
            return y.astype(np.float32)
        except Exception:
            return np.zeros(SR * 5, dtype=np.float32)

    def crop(self, y):
        if len(y) < WINDOW_SAMPLES:
            pad = WINDOW_SAMPLES - len(y)
            left = np.random.randint(0, pad + 1) if self.train_mode else pad // 2
            y = np.pad(y, (left, pad - left))
        elif len(y) > WINDOW_SAMPLES:
            start = np.random.randint(0, len(y) - WINDOW_SAMPLES + 1) if self.train_mode else (len(y) - WINDOW_SAMPLES) // 2
            y = y[start: start + WINDOW_SAMPLES]
        return y

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = self.load_audio(row["filename"])
        y = self.crop(y)
        m = np.abs(y).max()
        if m > 0: y = y / m
        label = np.zeros(N_CLASSES, dtype=np.float32)
        if row["primary_label"] in self.label2idx:
            label[self.label2idx[row["primary_label"]]] = 1.0
        sec = str(row.get("secondary_labels", "")).strip()
        if sec and sec != "[]" and sec != "nan":
            for s in sec.replace("[", "").replace("]", "").replace("'", "").split(","):
                s = s.strip()
                if s in self.label2idx:
                    label[self.label2idx[s]] = 1.0
        return torch.from_numpy(y), torch.from_numpy(label), 1.0  # source_weight = 1.0 (hard)


class PseudoSCDS(Dataset):
    """Pseudo dataset: each item = (file_id, chunk_idx) pair, returns 20s audio crop + soft label."""
    def __init__(self, pseudo_file_ids, pseudo_probs_k, train_sc_dir, max_chunks_per_file=12):
        self.pseudo_file_ids = pseudo_file_ids
        self.pseudo_probs_k = pseudo_probs_k
        self.train_sc_dir = Path(train_sc_dir)
        self.max_chunks_per_file = max_chunks_per_file
        # Build (file_idx, chunk_idx) index
        self.index = [(fi, ci) for fi in range(len(pseudo_file_ids)) for ci in range(max_chunks_per_file)]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        file_idx, chunk_idx = self.index[idx]
        fid = str(self.pseudo_file_ids[file_idx])
        fpath = self.train_sc_dir / f"{fid}.ogg"
        # Load 60s, crop centered 20s around chunk position
        try:
            y, _ = librosa.load(str(fpath), sr=SR, mono=True)
        except Exception:
            y = np.zeros(SR * 60, dtype=np.float32)
        # 60s audio: chunk_idx * 5s = chunk center, 20s window = ±10s around chunk
        chunk_center_sec = (chunk_idx + 0.5) * 5
        win_half = WINDOW_SEC / 2
        start = max(0, int(chunk_center_sec - win_half) * SR)
        end = start + WINDOW_SAMPLES
        if end > len(y):
            y = np.pad(y, (0, max(0, end - len(y))))
        y = y[start:end].astype(np.float32)
        if len(y) < WINDOW_SAMPLES:
            y = np.pad(y, (0, WINDOW_SAMPLES - len(y)))
        elif len(y) > WINDOW_SAMPLES:
            y = y[:WINDOW_SAMPLES]
        m = np.abs(y).max()
        if m > 0: y = y / m
        soft_label = self.pseudo_probs_k[file_idx, chunk_idx].astype(np.float32)
        return torch.from_numpy(y), torch.from_numpy(soft_label), PSEUDO_LOSS_WEIGHT


print("FocalDS + PseudoSCDS defined")


In [ ]:
# ★ V3: Beta MixUp (M7 流、V1 fixed-blend 100% から変更)
def mixup_audio_beta(wav, label, alpha=MIXUP_BETA_ALPHA, p=MIXUP_P):
    if np.random.random() >= p:
        return wav, label
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(wav.size(0), device=wav.device)
    return lam * wav + (1 - lam) * wav[idx], lam * label + (1 - lam) * label[idx]


class SpecAug(nn.Module):
    def __init__(self, freq=SPECAUG_FREQ, time_mask=SPECAUG_TIME):
        super().__init__()
        self.fa = T.FrequencyMasking(freq)
        self.ta = T.TimeMasking(time_mask)
    def forward(self, spec):
        return self.ta(self.fa(spec))


In [ ]:
def compute_per_species_auc(y_true, y_pred, mask=None, class_mask=None):
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    indices = range(y_true.shape[1]) if class_mask is None else class_mask
    aucs = []
    for c in indices:
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            auc = roc_auc_score(col, y_pred[:, c])
            aucs.append((int(c), float(auc)))
        except ValueError:
            continue
    return aucs


def macro_auc_from_list(aucs):
    return float(np.mean([a for _, a in aucs])) if len(aucs) > 0 else float("nan")


def lowest_k_mean(aucs, k=NS22_K):
    if len(aucs) == 0: return float("nan")
    sorted_aucs = sorted([a for _, a in aucs])
    k_eff = min(k, len(sorted_aucs))
    return float(np.mean(sorted_aucs[:k_eff]))


def class_stats_str(aucs):
    if len(aucs) == 0:
        return "n=0 median=nan p25=nan p75=nan #>0.5=0 #>0.7=0 #>0.9=0 #perfect=0"
    vals = np.array([a for _, a in aucs])
    return (f"n={len(vals)} median={np.median(vals):.3f} p25={np.percentile(vals,25):.3f} "
            f"p75={np.percentile(vals,75):.3f} "
            f"#>0.5={int((vals>0.5).sum())} #>0.7={int((vals>0.7).sum())} "
            f"#>0.9={int((vals>0.9).sum())} #perfect={int((vals>=1.0).sum())}")


def taxon_str(y_true, y_pred, mask=None):
    parts = []
    for t in ["Insecta", "Reptilia", "Amphibia", "Mammalia", "Aves"]:
        cmask = TAXON_MASKS[t]
        if len(cmask) == 0:
            parts.append(f"{t}=nan"); continue
        aucs = compute_per_species_auc(y_true, y_pred, mask=mask, class_mask=cmask)
        m = macro_auc_from_list(aucs)
        parts.append(f"{t}={m:.3f}" if not np.isnan(m) else f"{t}=nan")
    return "taxon: " + " ".join(parts)


# ★ V3: labeled SC val 用 waveform loader (20s window centered at chunk)
def _load_val_waveforms_20s(val_sc_df):
    """Load 20s waveforms centered at each labeled 5s chunk."""
    wavs = []
    for _, row in val_sc_df.iterrows():
        fn = str(row["filename"])
        fn_with_ext = fn if fn.endswith(".ogg") else fn + ".ogg"
        start_sec = int(row["start_sec"])
        path = TRAIN_SC_DIR / fn_with_ext
        try:
            y, _ = librosa.load(str(path), sr=SR, mono=True)
        except Exception:
            y = np.zeros(SR * 60, dtype=np.float32)
        chunk_center = start_sec + 2.5
        win_start_sec = max(0.0, chunk_center - WINDOW_SEC / 2)
        start_samples = int(win_start_sec * SR)
        end_samples = start_samples + WINDOW_SAMPLES
        if end_samples > len(y):
            y = np.pad(y, (0, max(0, end_samples - len(y))))
        wav = y[start_samples:end_samples].astype(np.float32)
        if len(wav) < WINDOW_SAMPLES:
            wav = np.pad(wav, (0, WINDOW_SAMPLES - len(wav)))
        elif len(wav) > WINDOW_SAMPLES:
            wav = wav[:WINDOW_SAMPLES]
        m = np.abs(wav).max()
        if m > 0: wav = wav / m
        wavs.append(torch.from_numpy(wav))
    return wavs


@torch.no_grad()
def evaluate_on_labeled_sc(model, val_wavs, Y_val, device, batch_size=16):
    """Predict on 20s val waveforms, blend clip + framewise_max sigmoid."""
    model.eval()
    preds = []
    n = len(val_wavs)
    for s in range(0, n, batch_size):
        batch = torch.stack(val_wavs[s:s+batch_size]).to(device)
        with autocast():
            clip_logit, framewise_logit = model(batch, return_framewise=True)
            frame_max = framewise_logit.max(dim=1).values
            p_clip = torch.sigmoid(clip_logit).float().cpu().numpy()
            p_fmax = torch.sigmoid(frame_max).float().cpu().numpy()
            p_blend = 0.5 * p_clip + 0.5 * p_fmax
        preds.append(p_blend)
    return np.concatenate(preds)


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR


def train_fold(fold_k):
    print(f"\n{'='*60}\n[Fold {fold_k}] M3' V4 training {N_EPOCHS} epochs (Babych init + Hybrid V2 + Beta MixUp + grad_clip)\n{'='*60}")
    t0_fold = time.time()

    tr_df = train_df[train_df["fold"] != fold_k].reset_index(drop=True)
    print(f"  train (focal): {len(tr_df)}")

    tr_focal = FocalDS(tr_df, LABEL2IDX, TRAIN_AUDIO_DIR, train_mode=True)
    pseudo_ds = PseudoSCDS(pseudo_file_ids, pseudo_probs_k, TRAIN_SC_DIR)
    print(f"  focal: {len(tr_focal)}, pseudo: {len(pseudo_ds)}")

    combined = ConcatDataset([tr_focal, pseudo_ds])
    weights = np.concatenate([
        np.full(len(tr_focal), 0.80 / max(len(tr_focal), 1)),
        np.full(len(pseudo_ds), 0.20 / max(len(pseudo_ds), 1)),
    ])
    sampler = WeightedRandomSampler(weights, num_samples=len(tr_focal), replacement=True)
    tr_dl = DataLoader(combined, batch_size=BATCH_SIZE, sampler=sampler,
                       num_workers=2, pin_memory=True, drop_last=True)

    if len(sc_meta) > 0:
        vm = sc_meta["fold"].values == fold_k
        val_sc_df = sc_meta[vm].reset_index(drop=True)
        Y_val = Y_SC[vm]
        ns22_val = non_s22_mask_sc[vm]
        print(f"  val (labeled SC): {len(val_sc_df)} chunks (ns22={ns22_val.sum()})")
        val_wavs = _load_val_waveforms_20s(val_sc_df)
    else:
        val_wavs = []
        Y_val = np.zeros((0, N_CLASSES), dtype=np.float32)
        ns22_val = np.zeros(0, dtype=bool)

    model = make_model_with_babych_init().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    warmup_iters = WARMUP_EPOCHS * len(tr_dl)
    total_iters = N_EPOCHS * len(tr_dl)
    sched_warmup = LinearLR(optimizer, start_factor=1/25, end_factor=1.0, total_iters=warmup_iters)
    sched_cosine = CosineAnnealingLR(optimizer, T_max=total_iters - warmup_iters, eta_min=LR_MIN)
    scheduler = SequentialLR(optimizer, schedulers=[sched_warmup, sched_cosine], milestones=[warmup_iters])
    scaler = GradScaler()

    best_ns22 = -1.0
    best_macro = -1.0
    history = []
    total_steps = len(tr_dl)
    nan_count = 0
    NAN_TOLERANCE = 50   # NaN が 50 step 連続したら fold 中断

    for epoch in range(N_EPOCHS):
        t0_ep = time.time()
        model.train()
        tr_loss_sum = 0.0
        bce_sum = 0.0
        n_seen = 0
        for step, (wav, label, src_weight) in enumerate(tr_dl):
            wav = wav.to(DEVICE, non_blocking=True)
            label = label.to(DEVICE, non_blocking=True)
            src_weight = src_weight.to(DEVICE, non_blocking=True).float()
            wav, label = mixup_audio_beta(wav, label)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                clip_logit, framewise_logit = model(wav, return_framewise=True)
                frame_max_logit = framewise_logit.max(dim=1).values
                loss_clip = F.binary_cross_entropy_with_logits(clip_logit, label, reduction="none").mean(dim=1)
                loss_frame = F.binary_cross_entropy_with_logits(frame_max_logit, label, reduction="none").mean(dim=1)
                bce_per_sample = 0.5 * loss_clip + 0.5 * loss_frame
                bce = (bce_per_sample * src_weight).mean()
                distill = torch.tensor(0.0, device=DEVICE)
                loss = bce + distill

            # ★ V4: NaN guard
            if torch.isnan(loss).any() or torch.isinf(loss).any():
                nan_count += 1
                optimizer.zero_grad(set_to_none=True)
                if nan_count >= NAN_TOLERANCE:
                    print(f"  ★ Aborting fold {fold_k}: {NAN_TOLERANCE} consecutive NaN/Inf at ep{epoch+1} step {step}")
                    break
                continue
            else:
                nan_count = 0

            scaler.scale(loss).backward()
            # ★ V4: grad_clip 1.0 (gradient explosion 防止、M1 spec 準拠)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tr_loss_sum += loss.item() * wav.size(0)
            bce_sum += bce.item() * wav.size(0)
            n_seen += wav.size(0)

            if step % STEP_LOG_INTERVAL == 0 or step == total_steps - 1:
                cur_lr = optimizer.param_groups[0]["lr"]
                print(f"  [ep{epoch+1} step {step}/{total_steps}] loss={loss.item():.4f} bce={bce.item():.4f} distill=0.0000 lr={cur_lr:.2e}")

        if nan_count >= NAN_TOLERANCE:
            print(f"  [Fold {fold_k}] ABORTED at epoch {epoch+1}")
            return best_ns22

        tr_loss = tr_loss_sum / max(n_seen, 1)
        bce_avg = bce_sum / max(n_seen, 1)
        distill_avg = 0.0

        if len(val_wavs) > 0:
            val_preds = evaluate_on_labeled_sc(model, val_wavs, Y_val, DEVICE, batch_size=16)
            per_species_all = compute_per_species_auc(Y_val, val_preds)
            val_macro = macro_auc_from_list(per_species_all)
            per_species_ns22 = compute_per_species_auc(Y_val, val_preds, mask=ns22_val)
            val_ns22 = lowest_k_mean(per_species_ns22, k=NS22_K)
            tax_line = taxon_str(Y_val, val_preds, mask=ns22_val)
            cls_line = class_stats_str(per_species_ns22)
        else:
            val_macro = val_ns22 = float("nan")
            tax_line = "taxon: (no val)"
            cls_line = "class: (no val)"

        ep_time = (time.time() - t0_ep) / 60
        total_time = (time.time() - START) / 60
        cur_lr = optimizer.param_groups[0]["lr"]
        is_best = (not math.isnan(val_ns22)) and (val_ns22 > best_ns22)
        best_tag = "[BEST] " if is_best else ""

        print(f"=== Ep {epoch+1}/{N_EPOCHS}: loss={tr_loss:.4f} (bce={bce_avg:.4f} distill={distill_avg:.4f}) "
              f"val_ns22={val_ns22:.4f} val_macro={val_macro:.4f} {best_tag}lr={cur_lr:.2e} "
              f"({ep_time:.1f}min, total {total_time:.1f}min) ===")
        print(f"    {tax_line}")
        print(f"    class: {cls_line}")

        if is_best:
            best_ns22 = val_ns22
            torch.save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                        "epoch": epoch, "val_ns22": val_ns22, "val_macro": val_macro},
                       OUT_DIR / f"m3_fold{fold_k}_ckpt_best.pth")
            print(f"    BEST saved val_ns22={val_ns22:.4f}")
        if (not math.isnan(val_macro)) and val_macro > best_macro:
            best_macro = val_macro

        history.append({
            "ep": epoch, "tr_loss": tr_loss, "bce": bce_avg, "distill": distill_avg,
            "val_ns22": val_ns22, "val_macro": val_macro,
            "lr": cur_lr, "ep_time_min": ep_time, "best": is_best,
        })

    torch.save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                "epoch": N_EPOCHS - 1, "history": history},
               OUT_DIR / f"m3_fold{fold_k}_ckpt_final.pth")
    with open(OUT_DIR / f"m3_fold{fold_k}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    print(f"[Fold {fold_k}] DONE in {(time.time()-t0_fold)/60:.1f}min, best_ns22={best_ns22:.4f}")
    return best_ns22


fold_results = {}
for fold_k in range(N_FOLDS):
    bns22 = train_fold(fold_k)
    fold_results[fold_k] = bns22
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nAll folds DONE: {fold_results}")
print(f"Total time: {(time.time()-START)/60:.1f} min")


In [ ]:
# ONNX export per fold (for blend NB inference)
import onnxruntime as ort

class _M3ONNXWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, mel_input):
        # mel_input: (B, 3, n_mels, T) — pre-computed mel
        feat = self.model.backbone(mel_input)[-1]
        head_output = self.model.head(feat)
        framewise_logit = head_output["framewise_logit"]
        clip_logit = framewise_logit.max(dim=-1).values
        return clip_logit, framewise_logit.permute(0, 2, 1)


print("=== ONNX export per fold ===")
for fold_k in range(N_FOLDS):
    ckpt = torch.load(OUT_DIR / f"m3_fold{fold_k}_ckpt_best.pth", map_location="cpu", weights_only=False)
    model = make_model_with_babych_init()
    model.load_state_dict(ckpt["model_state"], strict=False)
    model.eval()
    wrapper = _M3ONNXWrapper(model)
    # Dummy mel input: (1, 3, 224, ~512)
    n_tf = WINDOW_SAMPLES // HOP_LENGTH + 1
    dummy_mel = torch.randn(1, 3, N_MELS, n_tf)
    onnx_path = OUT_DIR / f"m3_fold{fold_k}.onnx"
    try:
        torch.onnx.export(wrapper, dummy_mel, str(onnx_path),
                          input_names=["mel"], output_names=["clip_logit", "framewise"],
                          dynamic_axes={"mel": {0: "batch"}, "clip_logit": {0: "batch"}, "framewise": {0: "batch"}},
                          opset_version=17, do_constant_folding=True)
        print(f"  fold {fold_k}: ONNX exported ({onnx_path.stat().st_size/1e6:.1f} MB)")
    except Exception as e:
        print(f"  fold {fold_k}: ONNX export FAILED ({str(e)[:100]})、fallback to .pth")


In [ ]:
with open(OUT_DIR / "m3_summary.json", "w") as f:
    json.dump({
        "n_classes": N_CLASSES,
        "n_folds": N_FOLDS,
        "n_epochs": N_EPOCHS,
        "backbone": "eca_nfnet_l0",
        "mel_window_sec": WINDOW_SEC,
        "init": "babych_iter3",
        "fold_best_ns22": fold_results,
        "total_time_min": (time.time() - START) / 60,
    }, f, indent=2)
print(f"OK M3 DONE: {sorted(OUT_DIR.glob('m3_*'))}")
